In [ ]:
import torch
import torch.nn as nn
from torchvision import models
from PIL import Image
from torch.utils.data import Dataset

transform = (
    models.EfficientNet_B0_Weights
    .DEFAULT
    .transforms()
)

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

UTA_ROOT = (
    PROJECT_ROOT
    / "processed"
    / "UTA"
    / "sequences"
)

rows = []

for subject_dir in sorted(UTA_ROOT.iterdir()):

    subject = subject_dir.name

    for cls in ["0", "5", "10"]:

        frame_dir = subject_dir / cls

        for img_path in frame_dir.glob("*.jpg"):

            rows.append({
                "path": str(img_path),
                "subject": subject,
                "label": int(cls)
            })

df = pd.DataFrame(rows)

print(df.head())
print(df.shape)

In [ ]:
label_map = {
    0: 0,
    5: 1,
    10: 2
}

df["label"] = df["label"].map(label_map)

In [ ]:
TEST_SUBJECTS = [
    "01","06","11","16","21",
    "26","31","36","41","46"
]

In [ ]:
df = df[
    ~df["subject"].isin(TEST_SUBJECTS)
]

In [ ]:
import random

random.seed(42)

subjects = sorted(
    df["subject"].unique()
)

val_subjects = random.sample(
    list(subjects),
    8
)

train_subjects = [
    s for s in subjects
    if s not in val_subjects
]

print("Train:", len(train_subjects))
print("Val:", len(val_subjects))

In [ ]:
train_df = df[
    df["subject"].isin(train_subjects)
]

val_df = df[
    df["subject"].isin(val_subjects)
]

print(train_df["label"].value_counts())
print(val_df["label"].value_counts())

In [ ]:
def sample_subject(df_sub):

    sampled = []

    for s in df_sub["subject"].unique():

        sub = df_sub[
            df_sub["subject"] == s
        ]

        for c in [0,1,2]:

            cls = sub[
                sub["label"] == c
            ]

            n = min(
                200,
                len(cls)
            )

            sampled.append(
                cls.sample(
                    n=n,
                    random_state=42
                )
            )

    return pd.concat(sampled)

train_df = sample_subject(train_df)
val_df = sample_subject(val_df)

print("Sampled train size:", len(train_df))
print("Sampled validation size:", len(val_df))

print("\nSampled train class counts:")
print(train_df["label"].value_counts().sort_index())

print("\nSampled validation class counts:")
print(val_df["label"].value_counts().sort_index())

In [ ]:
from PIL import Image
from torch.utils.data import Dataset

transform = (
    models.EfficientNet_B0_Weights
    .DEFAULT
    .transforms()
)

class UTAFrameDataset(Dataset):

    def __init__(
        self,
        df,
        transform
    ):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        img = Image.open(
            row["path"]
        ).convert("RGB")

        img = self.transform(img)

        label = int(
            row["label"]
        )

        return img, label

In [ ]:
train_dataset = UTAFrameDataset(
    train_df,
    transform
)

val_dataset = UTAFrameDataset(
    val_df,
    transform
)

print(len(train_dataset))
print(len(val_dataset))

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = models.efficientnet_b0(
    weights=None
)

model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(1280, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, 2)
)

model.load_state_dict(
    torch.load(
        PROJECT_ROOT
        / "models"
        / "checkpoints"
        / "best_cnn.pth",
        map_location=device
    )
)

In [ ]:
model.classifier[-1] = nn.Linear(
    512,
    3
)

In [ ]:
for param in model.parameters():
    param.requires_grad = False

for param in model.features[-2:].parameters():
    param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=1e-5
)

In [ ]:
sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model.classifier)


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)
from tqdm import tqdm
import copy

In [ ]:
def train_one_epoch(model, loader):

    model.train()

    running_loss = 0
    preds = []
    labels = []

    for x, y in tqdm(loader):

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        out = model(x)

        loss = criterion(out, y)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        pred = out.argmax(1)

        preds.extend(pred.cpu().numpy())
        labels.extend(y.cpu().numpy())

    loss = running_loss / len(loader)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(
        labels,
        preds,
        average="macro"
    )

    return loss, acc, f1

In [ ]:
def validate(model, loader):

    model.eval()

    running_loss = 0
    preds = []
    labels = []

    with torch.no_grad():

        for x, y in tqdm(loader):

            x = x.to(device)
            y = y.to(device)

            out = model(x)

            loss = criterion(out, y)

            running_loss += loss.item()

            pred = out.argmax(1)

            preds.extend(pred.cpu().numpy())
            labels.extend(y.cpu().numpy())

    loss = running_loss / len(loader)

    acc = accuracy_score(labels, preds)

    f1 = f1_score(
        labels,
        preds,
        average="macro"
    )

    report = classification_report(
        labels,
        preds,
        target_names=[
            "Alert",
            "Low Vigilant",
            "Drowsy"
        ],
        digits=4
    )

    return loss, acc, f1, report

In [ ]:
x, y = next(iter(train_loader))
print(x.shape)
print(y.shape)

In [ ]:
NUM_EPOCHS = 20
PATIENCE = 5

best_f1 = 0
counter = 0

best_model = None
for epoch in range(NUM_EPOCHS):

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")

    train_loss, train_acc, train_f1 = train_one_epoch(
        model,
        train_loader
    )

    val_loss, val_acc, val_f1, report = validate(
        model,
        val_loader
    )

    print(
        f"Train Loss: {train_loss:.4f}"
        f" | Train Acc: {train_acc:.4f}"
        f" | Train F1: {train_f1:.4f}"
    )

    print(
        f"Val Loss: {val_loss:.4f}"
        f" | Val Acc: {val_acc:.4f}"
        f" | Val F1: {val_f1:.4f}"
    )

    print(report)

    if val_f1 > best_f1:

        best_f1 = val_f1
        counter = 0

        best_model = copy.deepcopy(
            model.state_dict()
        )

        torch.save(
            best_model,
            PROJECT_ROOT
            / "models"
            / "checkpoints"
            / "best_cnn_3class.pth"
        )

        print("✅ Best model saved.")

    else:

        counter += 1
        print(f"No improvement ({counter}/{PATIENCE})")

        if counter >= PATIENCE:
            print("Early stopping.")
            break